# RDPro Section Planner

Analyze a Python geospatial script, infer RDPro APIs, map APIs to sections, and build per-section generation specs.

In [34]:
from pathlib import Path
import json
import sys
import importlib

PROJECT_ROOT = Path('/Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph').resolve()
NOTEBOOK_DIR = PROJECT_ROOT / 'Notebook'
CODEGEN_DIR = NOTEBOOK_DIR / 'rdpro_section_codegen'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

for mod in [
    'rdpro_section_codegen',
    'rdpro_section_codegen.models',
    'rdpro_section_codegen.analyzer',
    'rdpro_section_codegen.planner',
    'rdpro_section_codegen.section_specs',
]:
    if mod in sys.modules:
        del sys.modules[mod]

import rdpro_section_codegen
import rdpro_section_codegen.models
import rdpro_section_codegen.analyzer
import rdpro_section_codegen.planner
import rdpro_section_codegen.section_specs

importlib.reload(rdpro_section_codegen.models)
importlib.reload(rdpro_section_codegen.analyzer)
importlib.reload(rdpro_section_codegen.planner)
importlib.reload(rdpro_section_codegen.section_specs)
importlib.reload(rdpro_section_codegen)

from rdpro_section_codegen import (
    analyze_python_script,
    build_plan,
    llm_infer_task_type,
    map_apis_to_sections,
    build_section_specs,
)

PYTHON_INPUT = PROJECT_ROOT / 'python' / 'nlcd_clip_zonal.py'
print('PYTHON_INPUT:', PYTHON_INPUT)

PYTHON_INPUT: /Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/python/nlcd_clip_zonal.py


In [35]:
python_code = PYTHON_INPUT.read_text(encoding='utf-8', errors='ignore')
task_info = llm_infer_task_type(python_code, provider='openai', model='gpt-4o')
print('LLM task info:', task_info)

analysis = analyze_python_script(
    python_code,
    task_type=task_info['task_type'],
    task_label=task_info.get('task_label', ''),
)
plan = build_plan(analysis)

print(json.dumps(plan.to_dict(), indent=2))

LLM task info: {'task_type': 'zonal_stats', 'task_label': 'zonal class summary', 'reason': 'The script performs zonal statistics by calculating land use percentages for each polygon in a shapefile using a raster dataset, and outputs the results to CSV files.'}
{
  "task_type": "zonal_stats",
  "apis": [
    "geoTiff",
    "tile metadata",
    "pixel type",
    "shapefile",
    "raptorJoin",
    "csv output"
  ],
  "sections": [
    "LOAD_DATA",
    "TYPE_CHECK",
    "RASTER_VECTOR_JOIN",
    "ANALYTICS"
  ],
  "reasons": {
    "LOAD_DATA": "materialize configured raster/vector inputs from bound paths; selected APIs: geoTiff, shapefile",
    "TYPE_CHECK": "validate pixel/data assumptions introduced during load; selected APIs: tile metadata, pixel type",
    "RASTER_VECTOR_JOIN": "selected from APIs: raptorJoin",
    "ANALYTICS": "derive per_feature_tabular_summary from transformed_raster"
  },
  "analysis": {
    "task_type": "zonal_stats",
    "python_raster_load_type": "implicit",
   

## Reasoning IR

Inspect the reasoning-centered intermediate representation directly.


In [36]:
print('Workflow goal:', analysis.workflow_goal)
print('Output intent:', analysis.output_intent)
print('Scalability constraints:')
for item in analysis.scalability_constraints:
    print('  -', item)
print('\nSection goals:')
for key, value in analysis.section_goals.items():
    print(f'  {key}: {value}')
print('\nIntermediate contracts:')
for key, value in analysis.intermediate_contracts.items():
    print(f'  {key}: {value}')
print('\nDataflow lineage:')
for step in analysis.dataflow_lineage:
    print('  -', step)
print('\nFull semantic_ir:')
print(json.dumps(analysis.semantic_ir, indent=2))


Workflow goal: zonal class summary using raster and vector inputs
Output intent: write distributed tabular output as csv with persisted_file semantics
Scalability constraints:
  - avoid collect() and other driver-heavy materialization
  - avoid validation-only count() on large datasets
  - preserve reusable distributed intermediates across sections
  - prefer distributed raster-vector operations over local geometry loops
  - prefer distributed, overwrite-safe output behavior

Section goals:
  LOAD_DATA: materialize configured raster/vector inputs from bound paths
  TYPE_CHECK: validate pixel/data assumptions introduced during load
  SPATIAL_CHECK: verify CRS/alignment requirements only when spatial compatibility matters
  TRANSFORM: prepare raster inputs with vector boundary alignment, reprojection, clipping, or masking before downstream analysis
  ANALYTICS: derive per_feature_tabular_summary from transformed_raster
  OUTPUT: persist tabular output as csv

Intermediate contracts:
  LO

## API To Section Map

Inspect how selected APIs are assigned to generation sections.

In [37]:
section_map = map_apis_to_sections(analysis, plan.apis)
print(json.dumps(section_map, indent=2))

{
  "LOAD_DATA": [
    "geoTiff",
    "shapefile"
  ],
  "TYPE_CHECK": [
    "tile metadata",
    "pixel type"
  ],
  "SPATIAL_CHECK": [],
  "TRANSFORM": [],
  "RASTER_VECTOR_JOIN": [
    "raptorJoin"
  ],
  "ANALYTICS": []
}


## Section Specs

Build per-section generation specs from the analysis and plan.

In [38]:
section_specs = build_section_specs(
    analysis,
    plan,
    (CODEGEN_DIR / 'job_scaffold.scala').read_text(encoding='utf-8', errors='ignore'),
)
print(json.dumps(section_specs, indent=2))

{
  "task_type": "zonal_stats",
  "task_label": "zonal class summary",
  "apis": [
    "geoTiff",
    "tile metadata",
    "pixel type",
    "shapefile",
    "raptorJoin",
    "csv output"
  ],
  "sections": [
    "LOAD_DATA",
    "TYPE_CHECK",
    "RASTER_VECTOR_JOIN",
    "ANALYTICS"
  ],
  "section_api_map": {
    "LOAD_DATA": [
      "geoTiff",
      "shapefile"
    ],
    "TYPE_CHECK": [
      "tile metadata",
      "pixel type"
    ],
    "SPATIAL_CHECK": [],
    "TRANSFORM": [],
    "RASTER_VECTOR_JOIN": [
      "raptorJoin"
    ],
    "ANALYTICS": []
  },
  "inputs": {
    "raster_inputs": [
      "/Users/clockorangezoe/Downloads/Annual_NLCD_LndCov_2024_CU_C1V1/Annual_NLCD_LndCov_2024_CU_C_1V1_clip.tif"
    ],
    "vector_inputs": [
      "/Users/clockorangezoe/Documents/phd_projects/data/Vector/us/us.shp"
    ],
    "output_candidates": [],
    "requires_vector": true,
    "multi_raster": false,
    "needs_alignment": true,
    "operations": [
      "load_raster",
      "load_

In [39]:
print('Task type:', plan.task_type)
print('Task label:', plan.analysis.get('task_label', ''))
print('Capabilities:', plan.analysis.get('capabilities', []))
print('APIs:', plan.apis)
print('Sections:', plan.sections)
print('Reasons:')
for key, value in plan.reasons.items():
    print(f'  {key}: {value}')

Task type: zonal_stats
Task label: zonal class summary
Capabilities: ['vector_input', 'masking', 'tabular_output']
APIs: ['geoTiff', 'tile metadata', 'pixel type', 'shapefile', 'raptorJoin', 'csv output']
Sections: ['LOAD_DATA', 'TYPE_CHECK', 'RASTER_VECTOR_JOIN', 'ANALYTICS']
Reasons:
  LOAD_DATA: materialize configured raster/vector inputs from bound paths; selected APIs: geoTiff, shapefile
  TYPE_CHECK: validate pixel/data assumptions introduced during load; selected APIs: tile metadata, pixel type
  RASTER_VECTOR_JOIN: selected from APIs: raptorJoin
  ANALYTICS: derive per_feature_tabular_summary from transformed_raster


## Run Section Agent

Launch the section-by-section generate/compile/fix loop using the planned Python input and the scaffold template. The generated Scala is written to a separate output file, not the scaffold.

In [40]:
import subprocess

ENV_PYTHON = Path('/Users/clockorangezoe/miniconda3/envs/geo_llm_spark/bin/python')

SECTION_AGENT = CODEGEN_DIR / 'langgraph_section_agent.py'
SCAFFOLD_SCALA = CODEGEN_DIR / 'job_scaffold.scala'
OUTPUT_SECTIONAL = CODEGEN_DIR / 'nlcd_zonal_sectional.scala'
API_DOC = PROJECT_ROOT / 'Doc' / 'rdpro_api_doc_combined.md'
GUIDE = NOTEBOOK_DIR / 'RDProAgentLoop_perAPI_fix_migration_guide.md'

cmd = [
    str(ENV_PYTHON),
    str(SECTION_AGENT),
    '--python-input', str(PYTHON_INPUT),
    '--scaffold', str(SCAFFOLD_SCALA),
    '--output-scala', str(OUTPUT_SECTIONAL),
    '--api-doc', str(API_DOC),
    '--guide', str(GUIDE),
    '--provider', 'openai',
    '--model', 'gpt-4o',
]

print('Running command:')
print(' '.join(cmd))
result = subprocess.run(cmd, cwd=str(CODEGEN_DIR), text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print('STDERR:\n' + result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Section agent failed with return code {result.returncode}')
print('Generated Scala:', OUTPUT_SECTIONAL)


Running command:
/Users/clockorangezoe/miniconda3/envs/geo_llm_spark/bin/python /Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/langgraph_section_agent.py --python-input /Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/python/nlcd_clip_zonal.py --scaffold /Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/job_scaffold.scala --output-scala /Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/nlcd_zonal_sectional.scala --api-doc /Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Doc/rdpro_api_doc_combined.md --guide /Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/RDProAgentLoop_perAPI_fix_migration_guide.md --provider openai --model gpt-4o
{
  "success": true,
  "done": true,
  "input_mode": "python-script",
  "generated_python_reference": "",
  "section_idx": 4,
  "planned_sections": [
    "L